# Лабораторная №2 — бесшовный инференс U-Net на больших изображениях

**Цель:** обучить классическую U-Net на патчах 512×512 из Berlin AOI, продемонстрировать сетку швов при stride=512 и выполнить overlap-tile инференс с теоретическим RF=200 (stride=312). Сравнить 512 и 1024 по времени, числу окон, Accuracy и IoU классов «здания»/«дороги».

> Запуск: Google Colab → Runtime → Change runtime type → GPU. Ячейки выполняются сверху вниз. Результаты зависят от данных и обучения: заранее выдуманных значений метрик здесь нет. В каталоге `lab2_results` сохраняются изображения, таблицы и веса.

**Источник данных:** https://www.kaggle.com/datasets/cceekkigg/berlin-aoi-dataset

**Важно:** набор Kaggle может иметь разные структуры каталогов. Ниже есть автоматический поиск пар и ручная настройка путей; перед обучением обязательно сверяем цветовую разметку на изображении.

In [ ]:
%pip -q install kagglehub pillow matplotlib pandas tqdm scikit-learn

In [ ]:
import os, re, math, time, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix
import kagglehub

SEED=2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT=Path('lab2_results'); ROOT.mkdir(exist_ok=True)
TILE=512; EPOCHS=12; TRAIN_SAMPLES=640; VAL_SAMPLES=128
BATCH_SIZE=4 if torch.cuda.is_available() else 1
BASE_CHANNELS=16  # 16 для доступной VRAM; 4 уровня и все операции классические
print('Device:',DEVICE, 'PyTorch:',torch.__version__)

## 1. Загрузка Berlin AOI и проверка формата разметки

Данные скачиваются в среду Colab средствами KaggleHub. При ошибке авторизации можно загрузить архив вручную и указать `DATA_ROOT`. Автоматический поиск не подменяет проверку: он показывает обнаруженные пары и примеры цветов маски. Ожидаются классы 0 (фон, чёрный), 1 (здания, синий), 2 (дороги, белый).

In [ ]:
try:
    DATA_ROOT=Path(kagglehub.dataset_download('cceekkigg/berlin-aoi-dataset'))
except Exception as exc:
    print('KaggleHub download failed:', repr(exc))
    DATA_ROOT=Path('/content/berlin-aoi-dataset')  # распакуйте сюда архив вручную
print('DATA_ROOT:', DATA_ROOT)
assert DATA_ROOT.exists(), 'Датасет не найден. Проверьте доступ Kaggle или DATA_ROOT.'
files=[p for p in DATA_ROOT.rglob('*') if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}]
print('Image-like files:',len(files))
for p in files[:35]: print(p.relative_to(DATA_ROOT))

In [ ]:
def is_mask_path(p):
    s=str(p.relative_to(DATA_ROOT)).lower()
    return any(x in s for x in ('mask','label','annotation','ground_truth','groundtruth','_gt','/gt/','/y/'))
def canonical(p):
    name=p.stem.lower()
    for word in ('_mask','_masks','_label','_labels','_annotation','_gt','_rgb','_image','_img'):
        name=name.replace(word,'')
    return re.sub(r'[^a-z0-9]','',name)
mask_files=[p for p in files if is_mask_path(p)]
image_files=[p for p in files if not is_mask_path(p)]
by_id={}
for p in mask_files: by_id.setdefault(canonical(p),[]).append(p)
pairs=[(p,by_id[canonical(p)][0]) for p in image_files if canonical(p) in by_id]
# При иных названиях папок впишите пути вручную:
MANUAL_PAIRS=[]  # пример: [('/content/data/images/1.png','/content/data/masks/1.png')]
if MANUAL_PAIRS: pairs=[(Path(a),Path(b)) for a,b in MANUAL_PAIRS]
print('Matched image/mask pairs:',len(pairs))
for a,b in pairs[:12]: print(a.name,' -> ',b.name)
assert pairs, 'Не найдены пары автоматически. Изучите список файлов и задайте MANUAL_PAIRS.'
def read_rgb(p): return np.asarray(Image.open(p).convert('RGB'), dtype=np.uint8)
for a,b in pairs[:2]:
    x=read_rgb(a); y=np.asarray(Image.open(b).convert('RGB'))
    vals,cnt=np.unique(y.reshape(-1,3),axis=0,return_counts=True)
    print(a.name,x.shape,'| mask',y.shape,'| most common:',list(zip(vals[np.argsort(cnt)[-8:]].tolist(),np.sort(cnt)[-8:].tolist())))
    fig,ax=plt.subplots(1,2,figsize=(12,5));ax[0].imshow(x);ax[1].imshow(y);ax[0].set_title('Image');ax[1].set_title('Mask')
    for aa in ax: aa.axis('off')
    plt.show()

### Кодирование масок и разбиение по изображениям

В отличие от разбиения патчей случайным образом, **train/validation/test разбиваются по исходным снимкам**: патчи одного снимка не могут одновременно попасть в обучение и тест. Белые пиксели имеют класс 2, синие — 1. Если цвета файла отличаются, настройте `COLOR_MAP` после просмотра предыдущей ячейки.

In [ ]:
COLOR_MAP={(0,0,0):0,(0,0,255):1,(255,255,255):2}
def decode_mask(path):
    im=np.asarray(Image.open(path))
    if im.ndim==2:
        unique=np.unique(im)
        if set(unique.tolist()).issubset({0,1,2}): return im.astype(np.uint8)
        raise ValueError(f'Indexed mask labels {unique[:20]} in {path}; adjust decoder')
    rgb=np.asarray(Image.open(path).convert('RGB'),dtype=np.uint8)
    result=np.full(rgb.shape[:2],255,dtype=np.uint8)
    for color,cls in COLOR_MAP.items(): result[np.all(rgb==color,axis=-1)]=cls
    unknown=int((result==255).sum())
    if unknown:
        raise ValueError(f'{path.name}: {unknown} pixels with unknown colors. Inspect masks and COLOR_MAP; no silent relabeling.')
    return result
for a,b in pairs[:min(3,len(pairs))]:
    y=decode_mask(b); print(a.name, 'classes',np.unique(y,return_counts=True))
rng=np.random.default_rng(SEED)
order=rng.permutation(len(pairs))
assert len(pairs)>=3, 'Для независимого train/val/test требуется минимум 3 снимка.'
n_test=max(1,round(len(pairs)*0.2))
n_val=max(1,round(len(pairs)*0.2))
if n_test+n_val>=len(pairs): n_test=n_val=1
test_pairs=[pairs[i] for i in order[:n_test]]
val_pairs=[pairs[i] for i in order[n_test:n_test+n_val]]
train_pairs=[pairs[i] for i in order[n_test+n_val:]]
print('Images train / val / test:',len(train_pairs),len(val_pairs),len(test_pairs))

In [ ]:
class RandomPatchDataset(Dataset):
    def __init__(self,pairs,n_samples,tile=512,augment=False):
        self.pairs=pairs;self.n=n_samples;self.tile=tile;self.augment=augment
    def __len__(self):return self.n
    def __getitem__(self,index):
        # Файлы читаются по требованию, не кэшируем весь набор в RAM.
        a,b=self.pairs[index%len(self.pairs)]
        x=read_rgb(a);y=decode_mask(b)
        assert x.shape[:2]==y.shape, (a,b,x.shape,y.shape)
        h,w=y.shape;t=self.tile
        if h<t or w<t:
            pad=((0,max(0,t-h)),(0,max(0,t-w)))
            x=np.pad(x,(pad[0],pad[1],(0,0)),mode='reflect')
            y=np.pad(y,pad,mode='reflect')
            h,w=y.shape
        yy=np.random.randint(0,h-t+1);xx=np.random.randint(0,w-t+1)
        x=x[yy:yy+t,xx:xx+t];y=y[yy:yy+t,xx:xx+t]
        if self.augment:
            if random.random()<.5:x=np.fliplr(x);y=np.fliplr(y)
            if random.random()<.5:x=np.flipud(x);y=np.flipud(y)
            k=random.randint(0,3);x=np.rot90(x,k);y=np.rot90(y,k)
        return torch.from_numpy(np.ascontiguousarray(x.transpose(2,0,1))).float()/255,torch.from_numpy(np.ascontiguousarray(y)).long()
train_dl=DataLoader(RandomPatchDataset(train_pairs,TRAIN_SAMPLES,augment=True),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available())
val_dl=DataLoader(RandomPatchDataset(val_pairs,VAL_SAMPLES),batch_size=BATCH_SIZE,num_workers=2,pin_memory=torch.cuda.is_available())

## 2. Классическая U-Net: четыре down/up-блока

Две свёртки 3×3 на каждом уровне, MaxPool2d(2), nearest-neighbor upsampling, skip concatenation и 1×1 выход для трёх классов. Нормализации и attention нет, чтобы архитектура совпадала с расчётом RF методички.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self,c_in,c_out):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(c_in,c_out,3,padding=1),nn.ReLU(inplace=True),
                               nn.Conv2d(c_out,c_out,3,padding=1),nn.ReLU(inplace=True))
    def forward(self,x):return self.net(x)
class UNet(nn.Module):
    def __init__(self,base=16):
        super().__init__()
        self.enc=nn.ModuleList([DoubleConv(3,base),DoubleConv(base,base*2),
                                DoubleConv(base*2,base*4),DoubleConv(base*4,base*8)])
        self.pool=nn.MaxPool2d(2)
        self.bottleneck=DoubleConv(base*8,base*16)
        self.dec=nn.ModuleList([DoubleConv(base*24,base*8),DoubleConv(base*12,base*4),
                                DoubleConv(base*6,base*2),DoubleConv(base*3,base)])
        self.out=nn.Conv2d(base,3,1)
    def forward(self,x):
        skips=[]
        for block in self.enc:
            x=block(x);skips.append(x);x=self.pool(x)
        x=self.bottleneck(x)
        for block,skip in zip(self.dec,reversed(skips)):
            x=F.interpolate(x,scale_factor=2,mode='nearest')
            if x.shape[-2:]!=skip.shape[-2:]:
                x=F.interpolate(x,size=skip.shape[-2:],mode='nearest')
            x=block(torch.cat([skip,x],dim=1))
        return self.out(x)
model=UNet(BASE_CHANNELS).to(DEVICE)
print('Parameters:',sum(p.numel() for p in model.parameters()))
with torch.inference_mode(): print('Sanity output:',tuple(model(torch.rand(1,3,128,128,device=DEVICE)).shape))

## 3. Теоретическое рецептивное поле

Правила из методички: для Conv3×3 `RF += 2J`; для Pool2×2 `RF += J, J *= 2`; nearest upsampling уменьшает J вдвое, RF не меняет; skip connection берёт максимум RF ветвей. Это арифметическая модель RF из методички; при nearest upsampling точная зона зависимости отдельных пикселей может зависеть от их чётности, поэтому ниже используется консервативный margin=100.

In [ ]:
rf=1;jump=1;trace=[];skip_rf=[]
def record(stage): trace.append((stage,jump,rf))
record('Input')
for level in range(4):
    rf+=2*jump;record(f'Enc {level+1} conv1')
    rf+=2*jump;record(f'Enc {level+1} conv2')
    skip_rf.append(rf)
    rf+=jump;jump*=2;record(f'Pool {level+1}')
for k in range(2):rf+=2*jump;record(f'Bottleneck conv{k+1}')
for level,skip in enumerate(reversed(skip_rf),start=1):
    jump//=2;record(f'Upsample {level}')
    rf=max(rf,skip);record(f'Concat {level}')
    for k in range(2):rf+=2*jump;record(f'Dec {level} conv{k+1}')
record('Output 1x1')
rf_table=pd.DataFrame(trace,columns=['Операция','Jump','RF'])
display(rf_table)
RF=rf;MARGIN=math.ceil(RF/2)
assert (RF,MARGIN)==(200,100)
for tile in (512,1024):
    safe=tile-2*MARGIN
    print(f'T={tile}, RF={RF}, margin={MARGIN}, central={safe}, stride={safe}, overlap={2*MARGIN}')
assert 512-2*MARGIN>0

## 4. Обучение на патчах 512×512

Loss — CrossEntropyLoss. Сохраняем лучшие веса по validation loss. Можно повысить `EPOCHS` и `TRAIN_SAMPLES`, если метрики неудовлетворительные. Нельзя подменять реальную оценку придуманными числами.

In [ ]:
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
criterion=nn.CrossEntropyLoss()
scaler=torch.amp.GradScaler('cuda',enabled=DEVICE.type=='cuda')
history=[];best=float('inf')
for epoch in range(1,EPOCHS+1):
    model.train();train_loss=0
    for xb,yb in tqdm(train_dl,desc=f'Epoch {epoch}/{EPOCHS}',leave=False):
        xb=xb.to(DEVICE,non_blocking=True);yb=yb.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
            loss=criterion(model(xb),yb)
        scaler.scale(loss).backward();scaler.step(optimizer);scaler.update()
        train_loss+=loss.item()*len(xb)
    model.eval();val_loss=0
    with torch.inference_mode():
        for xb,yb in val_dl:
            xb=xb.to(DEVICE);yb=yb.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
                loss=criterion(model(xb),yb)
            val_loss+=loss.item()*len(xb)
    train_loss/=len(train_dl.dataset);val_loss/=len(val_dl.dataset)
    history.append({'epoch':epoch,'train_loss':train_loss,'val_loss':val_loss})
    print(f'epoch={epoch} train={train_loss:.4f} val={val_loss:.4f}')
    if val_loss<best:
        best=val_loss;torch.save(model.state_dict(),ROOT/'unet_berlin_best.pt')
pd.DataFrame(history).to_csv(ROOT/'training_history.csv',index=False)
pd.DataFrame(history).set_index('epoch').plot(title='Training / validation loss',grid=True)
plt.savefig(ROOT/'training_curve.png',dpi=150,bbox_inches='tight');plt.show()
model.load_state_dict(torch.load(ROOT/'unet_berlin_best.pt',map_location=DEVICE,weights_only=True));model.eval()

## 5. Память при прямом инференсе 4300×3200

Ниже оценивается **верхнеуровневый объём одновременно сохранённых активаций** конкретной модели через hooks на малом входе с масштабированием по площади. Это не универсальная оценка пика VRAM: PyTorch повторно использует буферы, а CUDA/cuDNN выделяет временные рабочие пространства. Также показываем объём входа и выходных логитов. Основное вычисление полного изображения не выполняем.

In [ ]:
H_REF,W_REF=3200,4300
input_mib=H_REF*W_REF*3*4/2**20
logit_mib=H_REF*W_REF*3*4/2**20
acts=[]
hooks=[]
def size_hook(_,inputs,output):
    if isinstance(output,torch.Tensor): acts.append(output.numel()*output.element_size())
for module in model.modules():
    if isinstance(module,(nn.Conv2d,nn.MaxPool2d,nn.Upsample)):
        hooks.append(module.register_forward_hook(size_hook))
with torch.inference_mode():
    _=model(torch.zeros(1,3,256,256,device=DEVICE))
for hook in hooks:hook.remove()
factor=(H_REF*W_REF)/(256**2)
estimated_activation_mib=sum(acts)*factor/2**20
weights_mib=sum(p.numel()*p.element_size() for p in model.parameters())/2**20
print(pd.DataFrame([{'Component':'Input RGB FP32','MiB':input_mib},
                    {'Component':'Output logits FP32','MiB':logit_mib},
                    {'Component':'Model parameters','MiB':weights_mib},
                    {'Component':'Sum of layer outputs (upper accounting)','MiB':estimated_activation_mib}]).to_string(index=False))
print('В реальном запуске memory_allocated/max_memory_allocated зависит от устройства, AMP и буферов.')
print('Полный кадр не подаётся в модель: вычисления выполняются только тайлами.')

## 6. Два способа tiled inference

1. **Без overlap:** неперекрывающиеся тайлы T×T, края исходного изображения достраиваются через `reflect`, используются все предсказания, в том числе у границ.
2. **Overlap-tile:** выходное изображение разбивается на сохраняемые центральные блоки `C=T-2m`; каждый блок предсказывается по входному тайлу T×T с контекстом `m` по четырём сторонам. На краю изображения недостающий контекст создаётся `reflect`.

Сборка в RAM (`uint8`), GPU получает по одному тайлу. Обе версии полностью покрывают произвольную форму изображения; проверка счётчика гарантирует отсутствие пропусков.

In [ ]:
def reflect_pad_image(img, top,bottom,left,right):
    # np.pad reflect для геометрически корректного обращения за пределы снимка.
    return np.pad(img,((top,bottom),(left,right),(0,0)),mode='reflect')

@torch.inference_mode()
def predict_tile(model,patch):
    tensor=torch.from_numpy(np.ascontiguousarray(patch.transpose(2,0,1))).unsqueeze(0).to(DEVICE,dtype=torch.float32)/255.
    with torch.autocast(device_type=DEVICE.type,enabled=DEVICE.type=='cuda'):
        logits=model(tensor)
    return logits.argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)

def tiled_predict(model,img,tile,overlap=False,margin=MARGIN):
    h,w=img.shape[:2]
    if overlap:
        assert tile>2*margin and tile%16==0
        stride=tile-2*margin
        # Для каждого центра x,y исходный фрагмент [x-margin : x+stride+margin].
        rows=math.ceil(h/stride);cols=math.ceil(w/stride)
        bottom=rows*stride-h+margin;right=cols*stride-w+margin
        padded=reflect_pad_image(img,margin,bottom,margin,right)
    else:
        assert tile%16==0
        stride=tile;rows=math.ceil(h/tile);cols=math.ceil(w/tile)
        padded=reflect_pad_image(img,0,rows*tile-h,0,cols*tile-w)
    output=np.empty((h,w),dtype=np.uint8)
    counts=np.zeros((h,w),dtype=np.uint8)
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    start=time.perf_counter()
    for row in tqdm(range(rows),desc=f'T={tile}, overlap={overlap}',leave=False):
        for col in range(cols):
            y=row*stride;x=col*stride
            patch=padded[y:y+tile,x:x+tile]
            assert patch.shape[:2]==(tile,tile), (patch.shape,tile)
            pred=predict_tile(model,patch)
            if overlap: pred=pred[margin:margin+stride,margin:margin+stride]
            vh=min(stride,h-y);vw=min(stride,w-x)
            output[y:y+vh,x:x+vw]=pred[:vh,:vw]
            counts[y:y+vh,x:x+vw]+=1
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    seconds=time.perf_counter()-start
    assert np.all(counts==1), 'Есть пропуски или многократные записи!'
    return output,{'tile':tile,'mode':'overlap RF' if overlap else 'stride=T',
                   'stride':stride,'patches':rows*cols,'seconds':seconds,'height':h,'width':w}

def calculate_metrics(target,pred):
    cm=confusion_matrix(target.reshape(-1),pred.reshape(-1),labels=[0,1,2])
    tp=np.diag(cm);union=cm.sum(1)+cm.sum(0)-tp
    iou=np.divide(tp,union,out=np.full(3,np.nan),where=union!=0)
    return {'accuracy':float(tp.sum()/cm.sum()),'IoU_buildings':float(iou[1]),
            'IoU_roads':float(iou[2]),'mIoU_present':float(np.nanmean(iou))}
def colorize(mask):
    palette=np.array([[0,0,0],[0,0,255],[255,255,255]],dtype=np.uint8)
    return palette[mask]
def seam_disagreement(pred,step,band=4):
    # Диагностическая статистика: доля переходов между классами около швов.
    # НЕ равна GT-метрике и не доказывает отсутствие швов.
    a=(pred[:,1:]!=pred[:,:-1]);b=(pred[1:,:]!=pred[:-1,:])
    xidx=np.arange(step,pred.shape[1],step)-1
    yidx=np.arange(step,pred.shape[0],step)-1
    v=float(a[:,xidx].mean()) if len(xidx) else float('nan')
    h=float(b[yidx,:].mean()) if len(yidx) else float('nan')
    return (v,h)

# Тест покрытия на изображении, не совпадающем по размерам с сеткой.
dummy=np.zeros((137,219,3),dtype=np.uint8)
class IdentityBackground:
    def __call__(self,x):
        b,_,h,w=x.shape
        return torch.stack((torch.ones((b,h,w),device=x.device),
                            torch.zeros((b,h,w),device=x.device),
                            torch.zeros((b,h,w),device=x.device)),dim=1)
test_pred,test_info=tiled_predict(IdentityBackground(),dummy,512,True)
assert test_pred.shape==(137,219) and np.all(test_pred==0)
print('Coverage test:',test_info)

## 7. Эксперименты: 512 и 1024, с overlap и без

Используем **один отложенный test-снимок** для всех четырёх конфигураций. Если он не 4300×3200, результаты всё равно считаются честно для его реального размера, а требуемая геометрия 4300×3200 отдельно рассчитывается формулой. Никакого растягивания разметки до нужного размера.

In [ ]:
print('Theoretical number of windows for H=3200, W=4300:')
for tile in (512,1024):
    for overlap in (False,True):
        stride=tile-2*MARGIN if overlap else tile
        print(f'T={tile}, overlap={overlap}, S={stride}, N={math.ceil(3200/stride)*math.ceil(4300/stride)}')
TEST_IMAGE,TEST_MASK=test_pairs[0]
image=read_rgb(TEST_IMAGE);target=decode_mask(TEST_MASK)
assert image.shape[:2]==target.shape, 'Image/mask shape mismatch'
print('Test image:',TEST_IMAGE,'shape=',image.shape)
results=[];predictions={}
for tile in (512,1024):
    for overlap in (False,True):
        label=f'tile{tile}_'+('overlap' if overlap else 'no_overlap')
        pred,stats=tiled_predict(model,image,tile,overlap)
        predictions[label]=pred
        metrics=calculate_metrics(target,pred)
        boundary=seam_disagreement(pred,stats['stride'])
        row=dict(stats,**metrics,seam_edge_vertical=boundary[0],seam_edge_horizontal=boundary[1],name=label)
        results.append(row)
        print(row)
        Image.fromarray(colorize(pred)).save(ROOT/f'{label}_prediction.png')
table=pd.DataFrame(results).set_index('name')
table.to_csv(ROOT/'comparison.csv')
display(table[['tile','mode','stride','patches','seconds','accuracy','IoU_buildings','IoU_roads','mIoU_present']].round(4))

## 8. Визуализация швов и итоговых масок

Сохраняем обзор исходника, ground truth и результатов, а также крупный фрагмент на стыке тайлов. Видимость швов зависит от обученности сети: их нельзя гарантировать на любых весах. Сопоставление IoU/Accuracy показывает, помогло ли устранение краевого контекста на реальных данных.

In [ ]:
fig,axs=plt.subplots(2,3,figsize=(18,11))
views=[('RGB',image),('Ground truth',colorize(target)),
       ('512 / stride 512',colorize(predictions['tile512_no_overlap'])),
       ('512 / stride 312',colorize(predictions['tile512_overlap'])),
       ('1024 / stride 1024',colorize(predictions['tile1024_no_overlap'])),
       ('1024 / stride 824',colorize(predictions['tile1024_overlap']))]
for ax,(title,im) in zip(axs.flat,views):
    ax.imshow(im);ax.set_title(title);ax.axis('off')
fig.tight_layout();fig.savefig(ROOT/'comparison_full.png',dpi=170);plt.show()
h,w=target.shape
x0=max(0,min(512-180,w-400));y0=max(0,min(512-180,h-400))
x1=min(w,x0+400);y1=min(h,y0+400)
fig,axs=plt.subplots(1,4,figsize=(17,5))
for ax,label in zip(axs,['tile512_no_overlap','tile512_overlap','tile1024_no_overlap','tile1024_overlap']):
    ax.imshow(colorize(predictions[label])[y0:y1,x0:x1])
    ax.set_title(label);ax.axis('off')
fig.tight_layout();fig.savefig(ROOT/'seams_zoom.png',dpi=200);plt.show()

## 9. Проверка геометрии и контекста

Для **T=512**: RF=200, margin=100, C=S=312, перекрытие 200; на снимке 4300×3200 — `ceil(4300/312)×ceil(3200/312)=14×11=154` тайла против 63.

Для **T=1024**: тот же RF и margin, C=S=824, перекрытие 200; на снимке 4300×3200 — `6×4=24` тайла против `5×4=20`.

Проверка «бесшовности» состоит из двух разных утверждений: (1) сохранённые центральные области имеют необходимый теоретический контекст вне внешнего края изображения, (2) реальный уровень качества зависит от обучения и проверяется метриками. На внешнем краю reflect-padding является оговоренным граничным условием. Точное равенство всех пикселей полноразмерному инференсу зависит также от фазы pooling/nearest при сдвиге окна. В этом эксперименте шаги 312/824 кратны 8, но не 16; следовательно, **пиксельное тождество полноразмерному U-Net не гарантируется даже при достаточном RF**. Ниже есть отдельная проверка расхождения с прямым инференсом на небольшом изображении.

In [ ]:
# Сравниваем на 1024×1024 по возможности с прямым запуском модели.
# При OOM тест аккуратно пропускается; результаты основной работы остаются в файлах.
test_small=image[:min(image.shape[0],1024),:min(image.shape[1],1024)]
if test_small.shape[:2]==(1024,1024):
    try:
        if DEVICE.type=='cuda':torch.cuda.empty_cache()
        full=predict_tile(model,test_small)
        overlap_small,_=tiled_predict(model,test_small,512,True)
        discrepancy=float((full!=overlap_small).mean())
        print('Direct vs overlap pixel disagreement on 1024x1024:',discrepancy)
        assert 0<=discrepancy<=1
    except torch.cuda.OutOfMemoryError:
        print('Direct 1024x1024 reference omitted: GPU OOM')
        torch.cuda.empty_cache()
else:
    print('Reference check skipped: test image smaller than 1024x1024.')

## 10. Выводы

После выполнения ноутбука используйте численные результаты из `comparison.csv` и изображения `comparison_full.png`/`seams_zoom.png` для отчёта:

- Скользящее окно 512 без перекрытия экономит вычисления, но использует краевые предсказания с неполным контекстом.
- Теоретический RF данной классической U-Net равен 200×200; безопасная центральная область 512-2×100 = 312.
- Overlap-tile при T=512 обрабатывает 154 тайла против 63 на размере 4300×3200; при T=1024 — 24 против 20.
- В таблице приведены фактически измеренные время, Accuracy и IoU зданий/дорог (после обучения), а не заранее заданные числа.
- На практике дополнительная точность зависит от качества модели и реализации; «швы полностью отсутствуют» необходимо подтверждать сохранёнными визуализациями и метриками, а не обещать заранее.

**Примечание:** выходные файлы находятся в `lab2_results`; чтобы скачать их одним архивом в Colab, выполните следующую ячейку.

In [ ]:
import shutil
archive=shutil.make_archive('lab2_results','zip',ROOT)
print('Created',archive)
try:
    from google.colab import files as colab_files
    colab_files.download(archive)
except ImportError:
    print('Archive available in current working directory.')